# Notebook 04 — The Transformer Block

**Goal:** Build the repeating unit inside every transformer — the decoder block.

```
input x
   │
   ├───────────────────────────┐
   ▼                           │  (residual)
 LayerNorm                     │
   ▼                           │
 Multi-Head Self-Attention     │
   ▼                           │
 + ◄───────────────────────────┘
   │
   ├───────────────────────────┐
   ▼                           │  (residual)
 LayerNorm                     │
   ▼                           │
 Feed-Forward Network          │
   ▼                           │
 + ◄───────────────────────────┘
   │
   ▼
output
```

---

### Contents
1. [Feed-forward network](#1)
2. [Residuals and LayerNorm](#2)
3. [Complete transformer block](#3)
4. [Stacking blocks](#4)
5. [Key takeaways](#5)


In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(42)

print('=' * 60)
print('  Notebook 04 — The Transformer Block')
print('=' * 60)
print(f'  PyTorch : {torch.__version__}')
print('=' * 60)


  Notebook 04 — The Transformer Block
  PyTorch : 2.11.0


<a id='1'></a>
## 1 — Feed-forward network

Attention mixes information **between** tokens.  The FFN processes each token
**independently** — think of it as a per-position MLP:

$$
\text{FFN}(x) = W_2 \, \text{GELU}(W_1 x + b_1) + b_2
$$

The inner dimension `d_ff` is typically `4 × d_model` (expand, then contract).


In [2]:
class FeedForward(nn.Module):
    """Position-wise FFN:  d_model → d_ff → d_model"""
    def __init__(self, d_model, d_ff, dropout=0.0):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)    # expand
        self.linear2 = nn.Linear(d_ff, d_model)    # contract
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear1(x)        # (batch, seq, d_ff)
        x = F.gelu(x)              # smooth nonlinearity (GPT-2 uses GELU)
        x = self.dropout(x)
        x = self.linear2(x)        # (batch, seq, d_model)
        return x


ff = FeedForward(d_model=16, d_ff=64)
x_test = torch.randn(1, 6, 16)
print(f'FFN input  : {tuple(x_test.shape)}')
print(f'FFN output : {tuple(ff(x_test).shape)}')
print(f'Internally : 16 → 64 → 16  (4× expansion)')


FFN input  : (1, 6, 16)
FFN output : (1, 6, 16)
Internally : 16 → 64 → 16  (4× expansion)


<a id='2'></a>
## 2 — Residuals and LayerNorm

Two design choices make deep transformers trainable:

- **Residual connection** — the sub-layer learns a *change* to add on top of the
  input, rather than reconstructing the whole representation
- **LayerNorm** — normalises activations across the feature dimension per token,
  keeping gradients stable

GPT-2 uses **pre-norm** ordering (LN *before* the sub-layer), which is more
stable than the original post-norm.


In [3]:
# ── LayerNorm demo ────────────────────────────────────────────────────────────
x_demo = torch.tensor([[2.0, 4.0, 6.0, 8.0]])
ln = nn.LayerNorm(4)

print('Before LayerNorm:', x_demo.numpy())
print('After  LayerNorm:', ln(x_demo).detach().numpy().round(3))
print()
print('Mean ≈ 0, Std ≈ 1  →  stable scale regardless of the original magnitude.')


Before LayerNorm: [[2. 4. 6. 8.]]
After  LayerNorm: [[-1.342 -0.447  0.447  1.342]]

Mean ≈ 0, Std ≈ 1  →  stable scale regardless of the original magnitude.


<a id='3'></a>
## 3 — Complete transformer block

Pre-norm decoder block:

```
x → LN → MHA → +x → LN → FFN → +x → output
```


In [4]:
# ── Reuse MultiHeadAttention from notebook 03 ────────────────────────────────
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model, self.n_heads = d_model, n_heads
        self.d_k = d_model // n_heads
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        b, s, _ = x.shape
        Q = self.W_q(x).view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(b, s, self.n_heads, self.d_k).transpose(1, 2)
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask, float('-inf'))
        w = F.softmax(scores, dim=-1)
        w = torch.nan_to_num(w, nan=0.0)
        out = self.dropout(w) @ V
        out = out.transpose(1, 2).contiguous().view(b, s, self.d_model)
        return self.W_o(out)


class TransformerBlock(nn.Module):
    """
    GPT-style decoder block (pre-norm).

    x → LN1 → MHA → +x → LN2 → FFN → +x
    """
    def __init__(self, d_model, n_heads, d_ff, dropout=0.0):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, mask=None):
        # ── Sub-layer 1: attention + residual ─────────────────────────────────
        x = x + self.attn(self.ln1(x), mask=mask)
        # ── Sub-layer 2: FFN + residual ───────────────────────────────────────
        x = x + self.ff(self.ln2(x))
        return x


In [5]:
# ── Test the block ────────────────────────────────────────────────────────────
d_model, n_heads, d_ff, seq_len = 16, 4, 64, 6

block = TransformerBlock(d_model, n_heads, d_ff)
x = torch.randn(2, seq_len, d_model)
mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
mask = mask.unsqueeze(0).unsqueeze(0)

y = block(x, mask=mask)

print(f'Input  : {tuple(x.shape)}')
print(f'Output : {tuple(y.shape)}')
print(f'Shape preserved: {x.shape == y.shape}')
print(f'Parameters     : {sum(p.numel() for p in block.parameters()):,}')


Input  : (2, 6, 16)
Output : (2, 6, 16)
Shape preserved: True
Parameters     : 3,216


<a id='4'></a>
## 4 — Stacking blocks

Because input and output shapes match, blocks compose trivially.


In [6]:
# ── Stack 3 blocks ────────────────────────────────────────────────────────────
stack = nn.ModuleList([
    TransformerBlock(d_model=16, n_heads=4, d_ff=64) for _ in range(3)
])

x = torch.randn(1, 6, 16)
for i, blk in enumerate(stack):
    x = blk(x, mask=mask)
    print(f'After block {i}: {tuple(x.shape)}')

total = sum(p.numel() for p in stack.parameters())
print(f'\nTotal params (3 blocks): {total:,}')
print('GPT-2 Small is this pattern repeated 12× at d_model=768.')


After block 0: (1, 6, 16)
After block 1: (1, 6, 16)
After block 2: (1, 6, 16)

Total params (3 blocks): 9,648
GPT-2 Small is this pattern repeated 12× at d_model=768.


<a id='5'></a>
## 5 — Key takeaways

| Component | Purpose |
|-----------|---------|
| **Multi-Head Attention** | Mixes information between token positions |
| **Feed-Forward Network** | Transforms each token independently (per-position MLP) |
| **LayerNorm (pre-norm)** | Stabilises activations before each sub-layer |
| **Residual connection** | Enables gradient flow and lets each block learn a delta |
| **GELU** | Smoother activation than ReLU; standard in GPT-2 |

Architecture at a glance:
```
x → LN → MHA → +x → LN → FFN → +x → output
```

**Next:** Notebook 05 trains a full GPT-style model and generates text.
